[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tunnel-ai/way/blob/main/notebooks/05_03_main_embeddings.ipynb)

# Module 5, NLP: Discovering Subfields From Text Alone

**Notebook:** `05_03_main_embeddings`

## What we're doing

We have a few thousand recent papers on **machine learning**. We don't know what's in them. We don't tell the algorithm anything about subfields, journals, authors, or topics, just the text of each abstract.

By the end of this notebook, we'll have:
1. **A 2-D map** showing where every paper sits relative to every other paper.
2. **Named clusters**, "computer vision," "language models," "reinforcement learning," etc., derived purely from word patterns.
3. **Representative papers** for each cluster, to verify the names match the contents.

This is the foundation of how research-paper recommenders, semantic search, and topic-modeling pipelines actually work. The same recipe scales to product reviews, customer support tickets, news articles, or any text corpus you want to *discover* the structure of.

## The recipe

| Step | Tool | What it does |
| --- | --- | --- |
| Fetch | OpenAlex API | grab abstracts |
| Encode | sentence-transformers | turn each abstract into a 384-dim vector |
| Cluster | KMeans | group nearby vectors |
| Project | UMAP | flatten to 2-D for the picture |
| Label | c-TF–IDF | pick the most distinctive terms per cluster |

Every step is interpretable. Every step produces visible output. Let's go.

## 0) Setup

Two non-default packages: `sentence-transformers` (for embeddings, ~80 MB model download) and `umap-learn` (for the 2-D projection).

In [ ]:
import os
import re
import time
import textwrap

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    import umap
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "sentence-transformers", "umap-learn"])
    from sentence_transformers import SentenceTransformer
    import umap

In [ ]:
DATA_DIR = "assets/data"
os.makedirs(DATA_DIR, exist_ok=True)
CACHE_PATH = os.path.join(DATA_DIR, "openalex_ml_corpus.csv")

BASE = "https://api.openalex.org/works"

# OpenAlex concept ID for "Machine learning"
CONCEPT_ID = "C119857082"
MAX_WORKS  = 2000
FROM_YEAR  = 2023

## 1) Fetch a corpus we don't yet know the structure of

We pull recent (2023+) machine-learning papers from OpenAlex. We don't filter further, we want a slice broad enough that real subfields exist within it. The fetch is cached, so re-runs are instant.

In [ ]:
def inverted_index_to_text(inv):
    """OpenAlex stores abstracts as token -> [positions]. Reconstruct text."""
    if not isinstance(inv, dict) or len(inv) == 0:
        return None
    max_pos = 0
    for _, positions in inv.items():
        if positions:
            max_pos = max(max_pos, max(positions))
    tokens = [""] * (max_pos + 1)
    for token, positions in inv.items():
        for p in positions:
            if 0 <= p < len(tokens) and tokens[p] == "":
                tokens[p] = token
    text = " ".join(t for t in tokens if t)
    return text if text.strip() else None


def fetch_ml_corpus(max_works=MAX_WORKS):
    rows = []
    cursor = "*"
    while len(rows) < max_works:
        params = {
            "per-page": 200,
            "cursor": cursor,
            "filter": (
                f"has_abstract:true,"
                f"from_publication_date:{FROM_YEAR}-01-01,"
                f"concept.id:{CONCEPT_ID}"
            ),
            "sort": "cited_by_count:desc",
        }
        r = requests.get(BASE, params=params, timeout=60)
        r.raise_for_status()
        payload = r.json()
        for work in (payload.get("results") or []):
            if len(rows) >= max_works:
                break
            abstract = inverted_index_to_text(work.get("abstract_inverted_index"))
            if not abstract or len(abstract) < 200:
                continue
            rows.append({
                "openalex_id": work.get("id"),
                "title": work.get("title"),
                "publication_date": work.get("publication_date"),
                "cited_by_count": work.get("cited_by_count", 0),
                "abstract": abstract,
            })
        cursor = payload.get("meta", {}).get("next_cursor")
        if not cursor:
            break
        time.sleep(0.15)
    return pd.DataFrame(rows)

In [ ]:
if os.path.exists(CACHE_PATH):
    df = pd.read_csv(CACHE_PATH)
    print(f"Loaded cached corpus: {len(df):,} papers")
else:
    print("Fetching from OpenAlex (one-time, ~2-3 min)...")
    df = fetch_ml_corpus()
    df.to_csv(CACHE_PATH, index=False)
    print(f"Fetched and cached {len(df):,} papers to {CACHE_PATH}")

df.head(3)

## 2) Light cleaning

We strip URLs, lowercase, and keep letters and hyphens. Nothing fancy, embeddings are robust to small text variations, so over-cleaning costs us more than it helps.

In [ ]:
url_pat = re.compile(r"https?://\S+|www\.\S+")
multi_space_pat = re.compile(r"\s+")

def clean_text(s):
    if not isinstance(s, str):
        return ""
    s = url_pat.sub(" ", s.strip())
    return multi_space_pat.sub(" ", s).strip()

df["text"] = df["abstract"].apply(clean_text)
print(f"Cleaned {len(df):,} abstracts. Avg length: {int(df['text'].str.len().mean())} chars.")

## 3) Encode each abstract as a vector

We use `all-MiniLM-L6-v2`, a small sentence-transformer that maps any short text to a **384-dimensional vector**. Two abstracts that *mean* similar things end up close in this space, even if they don't share many words.

This is the conceptual jump from TF–IDF: TF–IDF treats "regulation" and "governance" as unrelated tokens. Embeddings know they're neighbors.

In [ ]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")
emb = encoder.encode(
    df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print("Embedding matrix:", emb.shape, "  (n_papers, n_dims)")

## 4) Cluster with KMeans

We pick `k = 10` clusters. This is a knob you can turn, `k=6` gives broader groups, `k=15` gives finer ones. Ten is a sweet spot for ML papers: small enough to read, big enough to separate the obvious subfields.

In [ ]:
K = 10
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(emb)

print("Cluster sizes:")
print(df["cluster"].value_counts().sort_index())

## 5) Name the clusters with c-TF–IDF

Now we ask: *what's distinctive about each cluster's vocabulary?*

**c-TF–IDF** (class-based TF–IDF, the trick behind BERTopic) is the elegant answer:

1. Concatenate all abstracts in cluster $c$ into one giant pseudo-document.
2. Run TF–IDF across the $K$ pseudo-documents (one per cluster).
3. The top-weighted terms in pseudo-document $c$ are the words most *characteristic* of cluster $c$, frequent within it, rare in the others.

The result: human-readable labels for clusters that were discovered without any labels.

In [ ]:
# Build one mega-document per cluster
cluster_docs = [" ".join(df.loc[df["cluster"] == c, "text"]) for c in range(K)]

ctfidf = TfidfVectorizer(
    stop_words="english",
    min_df=2,
    ngram_range=(1, 2),
    max_features=20_000,
)
M = ctfidf.fit_transform(cluster_docs)
terms = ctfidf.get_feature_names_out()

TOP_N = 10
cluster_top_terms = {}
for c in range(K):
    row = M[c].toarray().ravel()
    top_idx = row.argsort()[::-1][:TOP_N]
    cluster_top_terms[c] = [terms[i] for i in top_idx]

print("Top distinctive terms per cluster:\n")
for c in range(K):
    print(f"Cluster {c} (n={int((df['cluster']==c).sum()):>4d}): {', '.join(cluster_top_terms[c])}")

### Read the lists out loud

You should be able to *name* most of these clusters in plain English just from the top terms, "computer vision," "large language models," "reinforcement learning," "medical imaging," "recommender systems," and so on.

If a cluster's terms feel mixed, that's also informative: it usually means either (a) `K` is too small and two real subfields got merged, or (b) the cluster is a genuine catch-all of cross-cutting methods papers.

Try setting `K = 15` and re-running cells 4–5 to see what splits.

## 6) Map the topic space (UMAP)

The clusters live in 384-D, which we can't see. **UMAP** projects them to 2-D in a way that *preserves local neighborhood structure*: papers that are nearby in the high-dimensional space stay nearby in the picture.

Coloring by cluster gives us a topic map of the entire corpus.

In [ ]:
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42,
)
coords = reducer.fit_transform(emb)
print("2D coords:", coords.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

cmap = plt.cm.tab10
for c in range(K):
    mask = (df["cluster"] == c).values
    short_label = ", ".join(cluster_top_terms[c][:2])
    ax.scatter(
        coords[mask, 0], coords[mask, 1],
        s=12, alpha=0.55,
        color=cmap(c % 10),
        label=f"{c}: {short_label}",
    )
    # Annotate cluster centroid with its number
    cx, cy = coords[mask, 0].mean(), coords[mask, 1].mean()
    ax.annotate(
        str(c), (cx, cy),
        fontsize=14, fontweight="bold",
        ha="center", va="center",
        bbox=dict(boxstyle="circle,pad=0.3", facecolor="white", edgecolor="black", alpha=0.85),
    )

ax.set_title(f"Subfields discovered from text alone, {len(df):,} ML papers, {K} clusters")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, framealpha=0.9)
plt.tight_layout()
plt.show()

## 7) Are the clusters real? Sample some papers

Top terms can mislead. The honest sanity check is to look at the *actual papers* closest to each cluster's center and see whether the titles match the label.

In [ ]:
centroids = kmeans.cluster_centers_  # in embedding space, K x 384

def closest_to_centroid(c, n=5):
    """Return the n abstracts whose embeddings are closest to cluster c's centroid."""
    mask = (df["cluster"] == c).values
    sub_emb = emb[mask]
    sub_idx = np.where(mask)[0]
    # Cosine similarity to centroid (embeddings are already normalized)
    centroid_norm = centroids[c] / (np.linalg.norm(centroids[c]) + 1e-12)
    sims = sub_emb @ centroid_norm
    top_local = sims.argsort()[::-1][:n]
    return df.iloc[sub_idx[top_local]]

for c in range(K):
    label = ", ".join(cluster_top_terms[c][:3])
    print(f"\n=== Cluster {c}, {label} ===")
    reps = closest_to_centroid(c, n=3)
    for _, row in reps.iterrows():
        title = textwrap.shorten(str(row["title"]), width=110, placeholder="…")
        print(f"  • {title}")

## 8) Cluster sizes, are any clusters dominating?

An imbalanced split tells us something: subfields with more papers in the corpus genuinely produce more research right now. That's a finding, not a bug.

In [ ]:
sizes = df["cluster"].value_counts().sort_index()
labels = [f"{c}: {', '.join(cluster_top_terms[c][:2])}" for c in range(K)]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(range(K), sizes.values, color=[plt.cm.tab10(c % 10) for c in range(K)])
ax.set_yticks(range(K))
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel("Number of papers")
ax.set_title("Cluster sizes")
for bar, n in zip(bars, sizes.values):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, f"{n}",
            va="center", fontsize=9)
plt.tight_layout()
plt.show()

## What we just did

Starting from raw abstracts and *no labels*, we built:

- a **2-D map** of an entire research field,
- **named subfields** that any ML practitioner can recognize,
- **representative papers** that confirm the names match the contents,
- a **size distribution** that quantifies which subfields are most active.

The same five-step recipe, embed, cluster, project, label, verify, works for any text corpus where you want to *discover* the structure rather than impose it:

- **Customer support tickets** → discover the recurring problem categories.
- **Product reviews** → discover what aspects customers actually care about.
- **News articles** → discover storylines as they emerge.
- **Internal documents** → discover knowledge silos.

## Knobs to play with

Try one of these and re-run from cell 4:

- **`K`** (number of clusters): higher splits subfields into specialties; lower merges them into broad themes.
- **`CONCEPT_ID`** (corpus): swap `C119857082` (Machine Learning) for `C41008148` (Computer Science), `C71924100` (Medicine), or any [OpenAlex concept](https://api.openalex.org/concepts).
- **`encoder`** model: `all-mpnet-base-v2` is bigger and more accurate; `paraphrase-multilingual-MiniLM-L12-v2` works on non-English text.
- **UMAP `n_neighbors`** and **`min_dist`**: lower values give tighter, more separated clusters in the picture.

The pipeline doesn't change. The story does.